<table style="width: 100%">
    <tr>
    <td style="width: 5% text-align:center">
    </td>
    <td style="width: 18% text-align:center">
        <h3><ins>Parameter 3-Level</ins></h3>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_3b.ipynb"><h3>Parameter 4-Level</h3></a>
    </td>
    <td style="width: 5% text-align:center">
        <a href="parameter_optimize_4-level_system.ipynb"><h3>$\rightarrow$</h3></a>
    </td>
    </tr>
</table>

---
<div style="text-align: right;font-size: 16px">Script by Karl Horn, Daniel Basilewitsch, and Michael Goerz</div>

# Parameter Optimization of Three-Wave Mixing in a Three-Level System

This notebook contains the code for running an optimization of three-wave mixing in a three-level system.
You will learn how to use the `NLopt` package (via `Optimization.jl`) in order to optimize a set of pulses by varying parameters such as the pulse durations and intensities with the goal of driving two three-level systems, describing the quantum states of the enantiomers of a chiral molecule, such that they end up in distinct final states.

To this end, the notebook consists of the following sections
* [Model](#Model)
* [Setup](#Setup)
* [Hamiltonian](#Hamiltonian)
* [Problem 0: Pulse parameterisation](#Problem-0---Pulse-parameterisation)
* [Initialise optimization](#Initialise-optimization)
* [Problem 1: Run optimization](#Problem-1---Run-optimization)
* [Analyse optimization results](#Analyse-optimization-results)

and should be executed from top to bottom, whilst solving the problems as and when they appear.

## Model

<img src="../figures/3-level_mod.svg" alt="Drawing" style="width: 800px;"/>
The model above shows two three level systems corresponding to the $+$(left), respectively $-$(right), enantiomer of a chiral molecule.
Each of the three levels is connected by an electric dipol transition, with the sign of the $\mu_{c}$ component of the dipole being the only distiguishing factor between the two enantiomers. Obtaining 'enantio selectivity' in this context means that the same sequence of pulses applied to both enantiomers sharing the same initial state will lead to two distinct final states.

## Setup

We start with importing the necessary python packages, among which are
* `QuantumPropagators`: Set up time dependent Hamiltonians and propagate them
* `OrdinaryDiffEq`: The backend to use for `QuantumPropagators.propagate`
* `Optimization` for optimization, with `OptimizationNLOpt` providing the connection to the `NLOpt` backend package.

In [ ]:
using QuantumPropagators: hamiltonian, propagate
using OrdinaryDiffEq
using Optimization, OptimizationNLopt

In [ ]:
using Plots

# Set up thicker default lines in plots
Plots.default(
    linewidth               = 2.0,
    foreground_color_legend = nothing,
    background_color_legend = RGBA(1, 1, 1, 0.8)
)

In [ ]:
# Some utilities for showing hints and solutions
include("exercise_3a_utils.jl");

To make our code slightly easier to read, we define a constant 𝕚 for the imaginary unit:

In [ ]:
const 𝕚 = 1im;  # "𝕚" can be typed by \bbi<tab>

## Hamiltonian
Next, we define the function `total_enantiomer_ham` that returns the Hamiltonian (in the rotating frame) for either of the two enantiomers, i.e.,

$$
H_{I}^{(\pm)}(t) = \sum_{i=1}^{3} H_{i}^{(\pm)}(t)
$$

with

$$
H_{1}^{(\pm)}(t)
=
- E_{1}(t) \mu_{b}^{(\pm)}
\begin{pmatrix}
  0 & e^{i \phi_{1}} & 0 \\
  e^{- i \phi_{1}} & 0 & 0 \\
  0 & 0 & 0 \\
\end{pmatrix},
\\
H_{2}^{(\pm)}(t)
=
- E_{2}(t) \mu_{a}^{(\pm)}
\begin{pmatrix}
  0 & 0 & 0 \\
  0 & 0 & e^{i \phi_{2}} \\
  0 & e^{- i \phi_{2}} & 0 \\
\end{pmatrix},
\\
H_{3}^{(\pm)}(t)
=
- E_{3}(t) \mu_{c}^{(\pm)}
\begin{pmatrix}
  0 & 0 & e^{i \phi_{3}} \\
  0 & 0 & 0 \\
  e^{- i \phi_{3}} & 0 & 0 \\
\end{pmatrix}
$$

and where

$$
E_{i}(t)
=
\frac{E_{i,0}}{2}
\left[\tanh(a (t - t_{i,1})) - \tanh(a (t - t_{i,2}))\right]
$$

for $i \in \left\{1,2,3\right\}$ are the envelopes of (microwave) pulses with frequencies $\omega, \delta \omega$ and $\omega + \delta \omega$.

In [ ]:
E(t; E₀, t₁, t₂, a) =  (E₀/2) * (tanh(a*(t-t₁)) - tanh(a*(t-t₂)));

The `total_enantiomer_ham` function gets three different sets of input parameters, which together specify the Hamiltonian and its control fields entirely. The three set are:

* `pulse_durations`: A list of the three durations $\Delta t_{i} = t_{i,2} - t_{i,1}$ of each of the three fields $E_{i0}(t)$. Field $i$ is assumed to start when field $i-1$ ends. The first field starts at $t_{1,1}=0$.
* `phis`: A list of the three real phases $\phi_{i}$ for each field.
* `Ei0s`: A list of the three real amplitudes $E_{i0}$ for each field.

It also uses the following keyword arguments:

* `sign`: The string `+` or `-` specifies which Hamiltonian, i.e., $H_{I}^{(+)}(t)$ or $H_{I}^{(-)}(t)$, is retuned.
* `a`: Numerical parameter that controls how smooth each field is turned on and off. The larger `a` becomes, the more the field shapes $E_{i0}(t)$ resemble a rectangle.

In [ ]:
function total_enantiomer_ham(pulse_durations, phis, Ei0s; sign, a)

    ϕ₁, ϕ₂, ϕ₃ = phis
    μ = (sign == "-" ? -1 : 1)

    H₁ = μ * [
             0     exp(𝕚*ϕ₁)  0
        exp(-𝕚*ϕ₁)     0      0
             0         0      0
    ]

    H₂ = μ * [
        0       0          0
        0       0      exp(𝕚*ϕ₂)
        0  exp(-𝕚*ϕ₂)      0
    ]

    H₃ = μ * [
              0      0  exp(𝕚*ϕ₃)
              0      0      0
         exp(-𝕚*ϕ₃)  0      0
    ]

    # times where pulses end
    T₁ = sum(pulse_durations[1:1])
    T₂ = sum(pulse_durations[1:2])
    T₃ = sum(pulse_durations[1:3])

    return hamiltonian(
        (H₁, t -> E(t; E₀=Ei0s[1], t₁=0.0, t₂=T₁, a)),
        (H₂, t -> E(t; E₀=Ei0s[2], t₁=T₁, t₂=T₂, a)),
        (H₃, t -> E(t; E₀=Ei0s[3], t₁=T₂, t₂=T₃, a)),
    )
end

We define the initial state as

$$
\Psi_{\pm}(0)
=
\begin{pmatrix}
  1 \\ 0 \\ 0
\end{pmatrix}.
$$

In [ ]:
# the initial state consists of three levels with population initially in the ground state
Ψ₀ = ComplexF64[1, 0, 0];

Our time grid is obtained by dividing $$t \in \left[0,1\right]$$ into 100 equal intervals (100 intervals means 101 points).

In [ ]:
tlist = collect(range(0, 1; length=101));

## Problem 0 - Pulse parameterisation

Familiarise yourself with the concept of pulse parameterisation. The pulses we will be looking at in this script are shaped as the difference between two hyperbolic tangent functions.
`E₀` controls the pulse amplitude, `a` controls how rectangular the pulse appears and `t₁` and `t₂` determine when the pulse starts and ends.
Try changing the arguments of `plot_parameterised_pulse`, such that the two curves match (an exact fit is difficult, aim for a calculated mismatch of below one).

In [ ]:
function plot_parameterised_pulse(tlist; E₀, a, t₁, t₂)
    pulse = t -> E(t; E₀, a, t₁, t₂)
    target_pulse = t -> 20 * exp(-20 * (t - 0.5)^2)
    mismatch = sum(abs.(pulse.(tlist) .- target_pulse.(tlist))) / length(tlist)
    plot(tlist, pulse , label="your pulse", color=(mismatch < 1 ? "green" : "blue"))
    plot!(tlist, target_pulse; label="target pulse", color="orange")
    annotate!(
        0, 20,
        ("mismatch: $(round(mismatch; digits=3))", 10, :left)
    )
end

plot_parameterised_pulse(tlist; E₀=3.14, a=500, t₁=0.15, t₂=0.85)

In [ ]:
#step0.hint

In [ ]:
#step0.solution

**Bonus**: After you go through the "Problem 1" below, come back here and use the `Optimization` package to determine even better parameters.

In [ ]:
#bonus.hint

In [ ]:
#bonus.solution

## Initialise optimization

Now we can turn towards optimization. To this end, we use the optimization methods provided by the [`NLOpt` package](https://nlopt.readthedocs.io/en/latest/). This package has a [Julia wrapper](https://github.com/JuliaOpt/NLopt.jl) which we could directly, but there is also a package [Optimization.jl](https://docs.sciml.ai/Optimization/stable/) that provides a common API for defining and solving optimization problems not just via `NLOpt` but also a dozen other optimization toolboxes. For simplicity, we also use the standard Nelder-Mead method here, although it should be noted that `NLOpt` provides a whole lot of different methods (as do the other toolboxes wrapped by Optimization.jl).

The Optimization.jl interface requires to provide a function (here `optimize_selectivity`) that takes a vector `x` of (real) control parameters as well as an object `constants` containing static parameters. This could be another vector, but a [`NamedTuple`](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) is often cleaner. Note that for less trivial problems, the function to be optimized would be a more general [`OptimizationFunction`](https://docs.sciml.ai/Optimization/stable/API/optimization_function/) object that also contains information for how to calculate gradients. See the [Optimization.jl manual](https://docs.sciml.ai/Optimization/stable/API/optimization_problem/) for details. Nelder-Mead being a "gradient-free" method, we will not need this here.

Our function `optimize_selectivity` gets the list `x`, which contains our optimization parameters, i.e., all pulse durations, all phases $\phi_{i}$ and all amplitudes $E_{i,0}$, on input and returns the error of the enantio-selective process. In detail, `optimize_selectivity` returns zero if and only if the dynamics (governed by the parameters from set `x`) transfer the initial state $\Psi(0)$ into the respective target states, i.e.,

$$
\Psi_{+}(0)
\longrightarrow
\Psi_{+}\left(T\right) =
\begin{pmatrix}
  1 \\ 0 \\ 0
\end{pmatrix}
$$

for enantiomer `+` and

$$
\Psi_{-}(0)
\longrightarrow
\Psi_{-}\left(T\right) =
\begin{pmatrix}
  0 \\ 0 \\ 1
\end{pmatrix}
$$

for enantiomer `-`.

In [ ]:
obtained_fidelities = Float64[];  # for keeping track of the fidelity in each iteration

function optimize_selectivity(x, constants)

    global obtained_fidelities
    
    H₊, H₋ = opt_hams(x, constants)
    
    Ψ₀ = ComplexF64[1, 0, 0];
    tlist = collect(range(0, constants.T; length=constants.nt));

    Ψ₊ = propagate(Ψ₀, H₊, tlist; method=OrdinaryDiffEq)
    Ψ₋ = propagate(Ψ₀, H₋, tlist; method=OrdinaryDiffEq)

    fid = (abs(Ψ₊[1]) + abs(Ψ₋[3])) / 2
    push!(obtained_fidelities, fid)
    print("Iteration: $(length(obtained_fidelities)), current fidelity $(round(fid; digits=4))\r")
    return 1 - fid
    
end

In [ ]:
function opt_hams(control_parameters, constants)

    @assert length(control_parameters) == 9
    
    ΔT = control_parameters[1:3]
    ϕ = control_parameters[4:6]
    E₀ = control_parameters[7:9]
    a = constants.a  # we assume `constants` is a NamedTuple
    
    H₊ = total_enantiomer_ham(ΔT, ϕ, E₀; sign="+", a)
    H₋ = total_enantiomer_ham(ΔT, ϕ, E₀; sign="-", a)

    return H₊, H₋

end

## Problem 1 - Run optimization

Next, we can run the actual optimization. It requires us to define lower and upper bounds for all parameters that should be optimized. In our case, we choose

$$
0 \leq t_{i,2} \leq T=1,
\qquad
0 \leq \phi_{i} \leq 2 \pi,
\qquad
0 \leq E_{i,0} \leq 10.
$$

Note that we need to provide guess values for all nine parameters. The order that the parameters should appear in is:
* `pulse_durations`
* `phis`
* `Ei0s`

Their choice will have a significant impact on the general success and form of the optimized solution. Your task is fill in the upper optimization bounds and to try different guesses and evaluate their impact on the optimization.

In [ ]:
obtained_fidelities = Float64[];
guess = [#= insert guess parameters here =#]
prob = OptimizationProblem(
    optimize_selectivity,
    guess,
    (a=1000.0, T=1, nt=100);  # this is a NamedTuple
    lb=zeroes(9),
    ub=[#= insert upper bounds here here =#],
    stopval=(1-0.99),
);

In [ ]:
#step1.hint

In [ ]:
#step1.solution

We can check the quality of the guess pulse:

In [ ]:
optimize_selectivity(guess, (a=1000.0, T=tlist[end], nt=length(tlist)))

And the continue with the full optimization:

In [ ]:
obtained_fidelities = Float64[];
res = Optimization.solve(prob, NLopt.LN_NELDERMEAD(), maxiters=500)

In [ ]:
plot(obtained_fidelities; marker=:cross, label="", xlabel="optimization iteration", ylabel="fidelity")

## Analyse optimization results
After optimiziation we can verify the optimization result by plotting the pulses
as well as the resulting dynamics.
To this end, we define the Hamiltonians of the two enantiomers given the optimized parameters.

In [ ]:
H₊opt, H₋opt = opt_hams(res.u, (a=1000.0,));

In order to visualize the optimized pulses, we plot them in the following. We'll use

In [ ]:
using QuantumPropagators.Controls: get_controls

to extract the functions with the optimized parameters from the optimal Hamiltonians.

In [ ]:
function plot_pulse(H, tlist; kwargs...)
    fig = plot(; xlabel="time", ylabel="amplitude", kwargs...)
    sub = ["₁", "₂", "₃"]
    for (i, E) in enumerate(get_controls(H))
        plot!(fig, tlist, E; label="E$(sub[i])")
    end
    return fig
end

plot_pulse(H₊opt, range(0, 1; length=500); legend=:right)

Lastly, we solve the dynamics of the two enantiomers (now using the optimized parameters) and plot their population dynamics.

In [ ]:
function propagate_system(Htot, title; tlist=tlist)
    Ψ₀ = ComplexF64[1, 0, 0];
    states = propagate(Ψ₀, Htot, tlist; method=OrdinaryDiffEq, storage=true)
    pops = abs2.(states)
    plot(; title, xlabel="time", ylabel="population", legend=:top)
    plot!(tlist, pops[1,:]; label="|c₁|²")
    plot!(tlist, pops[2,:]; label="|c₂|²")
    plot!(tlist, pops[3,:]; label="|c₃|²")
end

In [ ]:
propagate_system(H₊opt, "enantiomer +")

In [ ]:
propagate_system(H₋opt, "enantiomer -")

In the plots above you see how the population evolves under the influence of our parameterised pulses. If the optimization was successful, the populations of the two enantiomers at the final time $T$ should be entirely in the $\Psi_{+}(T)$ (first level) and $\Psi_{-}(T)$ (third level) target states respectively.

---
<table style="width: 100%">
    <tr>
    <td style="width: 5% text-align:center">
    </td>
    <td style="width: 18% text-align:center">
        <h3><ins>Parameter 3-Level</ins></h3>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_4b.ipynb"><h3>Parameter 4-Level</h3></a>
    </td>
    <td style="width: 5% text-align:center">
        <a href="parameter_optimize_4-level_system.ipynb"><h3>$\rightarrow$</h3></a>
    </td>
    </tr>
</table>

---